# CareerPilot AI — Resume Parsing Pipeline

Run every cell top to bottom, in order. Each step prints its own output so you can see exactly what happened before moving to the next one.

**What this notebook does:** upload a resume -> extract its text -> parse name/skills with spaCy -> save the structured profile to SQLite so the rest of the team can use it.

## Step 0 — Install and import everything

In [1]:
!pip install pdfplumber python-docx spacy ipywidgets --quiet
!python -m spacy download en_core_web_sm --quiet

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [2]:
import os
import json
import sqlite3
import pdfplumber
from docx import Document
import spacy
from spacy.matcher import PhraseMatcher
import ipywidgets as widgets
from IPython.display import display

UPLOAD_DIR = "uploads"
os.makedirs(UPLOAD_DIR, exist_ok=True)

print("Setup complete.")

Setup complete.


## Step 1 — Upload your resume

Run the cell below. A button will appear — click **Upload** and choose your resume (PDF or DOCX).

If the button doesn't appear or doesn't work in your notebook environment, skip to the **manual fallback** in the cell after — just drag your resume file into the file browser panel on the left side of the notebook, then set the path by hand.

In [3]:
uploader = widgets.FileUpload(accept=".pdf,.docx", multiple=False)
display(uploader)

FileUpload(value=(), accept='.pdf,.docx', description='Upload')

In [ ]:
# Run this AFTER selecting a file in the cell above.
# Handles both old and new versions of ipywidgets automatically.

file_path = None

if len(uploader.value) > 0:
    uploaded = uploader.value
    if isinstance(uploaded, tuple):          # ipywidgets 8+
        item = uploaded[0]
        name = item["name"]
        content = item["content"]
    else:                                     # ipywidgets 7.x
        name = list(uploaded.keys())[0]
        content = uploaded[name]["content"]

    file_path = os.path.join(UPLOAD_DIR, name)
    with open(file_path, "wb") as f:
        f.write(bytes(content))

    print(f"Saved: {file_path}")
else:
    print("No file uploaded yet. Run the upload cell above, choose a file, then re-run this cell.")

# --- Manual fallback ---
# If the widget doesn't work, drag your resume into the file browser on the left,
# then comment out the block above and uncomment this line instead (edit the filename):
# file_path = "uploads/your_resume.pdf"

## Step 2 — Extract raw text from the file

PDFs and DOCX files store text differently, so we need a separate reader for each.

In [7]:
file_path = "uploads/koti resumer.docx"

In [8]:
def extract_text(path: str) -> str:
    if not path or not os.path.exists(path):
        raise ValueError(
            "No resume file found. Go back to Step 1: upload a file, or "
            "set file_path manually in the fallback line, then re-run this cell."
        )

    lower_path = path.lower()  # so .PDF / .DOCX (any case) are recognized too

    if lower_path.endswith(".pdf"):
        pages_text = []
        with pdfplumber.open(path) as pdf:
            for page in pdf.pages:
                pages_text.append(page.extract_text() or "")
        # join with a newline so the last word of one page doesn't fuse with
        # the first word of the next (which broke NER and skill matching)
        return "\n".join(pages_text)
    elif lower_path.endswith(".docx"):
        doc = Document(path)
        return "\n".join(p.text for p in doc.paragraphs)
    else:
        raise ValueError("Unsupported file type — use PDF or DOCX")

raw_text = extract_text(file_path)
print(raw_text[:500])

M. Koteswara Rao
Phone: 70938 57339 | Email: mannemkoteswararao48@gmail.com

Career Objective
Enthusiastic and detail-oriented B.Tech CSE (AI & ML) student with a strong foundation in programming, web development, and database management. Eager to contribute technical expertise, problem-solving skills, and a passion for learning to an entry-level software development or data-related role in the IT industry.
Education
Technical Skills
Programming Languages: Python, Object-Oriented Programming (OO


## Step 3 — Parse name, organizations, and locations with spaCy

spaCy's Named Entity Recognition (NER) scans the text and tags spans it recognizes as people, organizations, or places.

In [9]:
nlp = spacy.load("en_core_web_sm")

def get_entities(text: str) -> dict:
    doc = nlp(text)
    name = None
    organizations = []
    locations = []
    for ent in doc.ents:
        if ent.label_ == "PERSON" and name is None:
            name = ent.text
        elif ent.label_ == "ORG":
            organizations.append(ent.text)
        elif ent.label_ == "GPE":
            locations.append(ent.text)
    return {"name": name, "organizations": organizations, "locations": locations}

entities = get_entities(raw_text)
entities

{'name': 'M. Koteswara Rao\nPhone',
 'organizations': ['CSE',
  'AI & ML',
  'Object-Oriented Programming',
  'SQL',
  'Relational Database Management Systems',
  'JavaScript',
  'Student Database Management System',
  'SQL',
  'CRUD',
  'ORACLE\nPython',
  'Achievements & Activities\nActive',
  'Technical Clubs',
  'AI',
  'Strengths\nQuick'],
 'locations': []}

## Step 4 — Match known skills

This is a plain keyword list — expand it with more skills relevant to your test resumes as you go. Matching is case-insensitive.

In [10]:
SKILLS = [
    "Python", "Java", "JavaScript", "SQL", "React", "FastAPI", "Flask",
    "Machine Learning", "Deep Learning", "AWS", "Docker", "Kubernetes",
    "Git", "spaCy", "MongoDB", "PostgreSQL", "TensorFlow", "PyTorch",
    "Node.js", "HTML", "CSS", "C++", "Data Analysis", "NLP",
]

SKILL_LOOKUP = {skill.lower(): skill for skill in SKILLS}  # canonical casing

matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
matcher.add("SKILLS", [nlp.make_doc(skill) for skill in SKILLS])

def get_skills(text: str) -> list:
    doc = nlp(text)
    matches = matcher(doc)
    # normalize back to the canonical spelling from SKILLS, regardless of
    # how it was capitalized in the resume (e.g. "PYTHON" -> "Python")
    found = {SKILL_LOOKUP[doc[start:end].text.lower()] for _, start, end in matches}
    return sorted(found)

skills = get_skills(raw_text)
skills

['JavaScript', 'Python', 'SQL']

## Step 5 — Combine everything into one structured profile

In [11]:
profile = {
    "filename": os.path.basename(file_path),
    "name": entities["name"],
    "organizations": entities["organizations"],
    "locations": entities["locations"],
    "skills": skills,
    "raw_text": raw_text,
}

print(json.dumps(profile, indent=2))

{
  "filename": "koti resumer.docx",
  "name": "M. Koteswara Rao\nPhone",
  "organizations": [
    "CSE",
    "AI & ML",
    "Object-Oriented Programming",
    "SQL",
    "Relational Database Management Systems",
    "JavaScript",
    "Student Database Management System",
    "SQL",
    "CRUD",
    "ORACLE\nPython",
    "Achievements & Activities\nActive",
    "Technical Clubs",
    "AI",
    "Strengths\nQuick"
  ],
  "locations": [],
  "skills": [
    "JavaScript",
    "Python",
    "SQL"
  ],
  "raw_text": "M. Koteswara Rao\nPhone: 70938 57339 | Email: mannemkoteswararao48@gmail.com\n\nCareer Objective\nEnthusiastic and detail-oriented B.Tech CSE (AI & ML) student with a strong foundation in programming, web development, and database management. Eager to contribute technical expertise, problem-solving skills, and a passion for learning to an entry-level software development or data-related role in the IT industry.\nEducation\nTechnical Skills\nProgramming Languages: Python, Object-Or

## Step 6 — Set up the database schema

One table, with nested data (education, skills, experience) stored as JSON text inside columns rather than split across many linked tables. Simpler to build and just as functional for a hackathon timeline — this was the mentor's own recommendation.

This is the file the rest of your team (matching engine, roadmap generator) will read from.

In [12]:
import re

DB_PATH = "careerpilot.db"

def get_connection():
    return sqlite3.connect(DB_PATH)

def init_db():
    conn = get_connection()
    try:
        cur = conn.cursor()
        cur.execute('''
            CREATE TABLE IF NOT EXISTS career_profiles (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                filename TEXT,
                name TEXT,
                email TEXT,
                phone TEXT,
                location TEXT,
                education TEXT,
                skills TEXT,
                experience TEXT,
                organizations TEXT,
                certifications TEXT,
                career_goal TEXT,
                profile_completion INTEGER,
                raw_text TEXT,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
        ''')
        conn.commit()
    finally:
        conn.close()

init_db()
print("Database ready.")

Database ready.


### Functions to save, calculate completion, and read a profile back

In [13]:
def calculate_completion(p: dict) -> int:
    fields = ["name", "email", "phone", "location", "education", "skills", "experience", "career_goal"]
    filled = sum(1 for f in fields if p.get(f))
    return int((filled / len(fields)) * 100)

def save_profile(p: dict) -> int:
    conn = get_connection()
    try:
        cur = conn.cursor()
        completion = calculate_completion(p)
        cur.execute('''
            INSERT INTO career_profiles
            (filename, name, email, phone, location, education, skills, experience,
             organizations, certifications, career_goal, profile_completion, raw_text)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', (
            p.get("filename"),
            p.get("name"),
            p.get("email"),
            p.get("phone"),
            p.get("location"),
            json.dumps(p.get("education", [])),
            json.dumps(p.get("skills", [])),
            json.dumps(p.get("experience", [])),
            json.dumps(p.get("organizations", [])),
            json.dumps(p.get("certifications", [])),
            json.dumps(p.get("career_goal", {})),
            completion,
            p.get("raw_text", ""),
        ))
        conn.commit()
        return cur.lastrowid
    finally:
        conn.close()

def get_profile(profile_id: int) -> dict:
    conn = get_connection()
    try:
        cur = conn.cursor()
        cur.execute("SELECT * FROM career_profiles WHERE id = ?", (profile_id,))
        row = cur.fetchone()
        cols = [d[0] for d in cur.description]
    finally:
        conn.close()
    if row is None:
        return None
    data = dict(zip(cols, row))
    for f in ["education", "skills", "experience", "organizations", "certifications"]:
        data[f] = json.loads(data[f]) if data[f] else []
    data["career_goal"] = json.loads(data["career_goal"]) if data["career_goal"] else {}
    return data

print("Schema functions ready.")

Schema functions ready.


### Pull out email and phone with quick regex, then save

The parser from Steps 3-5 gives us name, organizations, locations, and skills — but not contact info, education, or experience yet. Email and phone are easy to grab with regex. Education and experience are left empty for now — a real, honest limitation of a simple parser, worth mentioning to your mentor rather than hiding.

In [14]:
def extract_contact_info(text: str) -> dict:
    email = re.search(r'[\w\.-]+@[\w\.-]+\.\w+', text)
    phone = re.search(r'(\+?\d{1,3}[-.\s]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}', text)
    return {
        "email": email.group() if email else None,
        "phone": phone.group() if phone else None,
    }

contact = extract_contact_info(raw_text)

full_profile = {
    "filename": profile["filename"],
    "name": profile["name"],
    "email": contact["email"],
    "phone": contact["phone"],
    "location": profile["locations"][0] if profile["locations"] else None,
    "organizations": profile["organizations"],
    "skills": profile["skills"],
    "education": [],       # not extracted yet — known limitation
    "experience": [],      # not extracted yet — known limitation
    "certifications": [],
    "career_goal": {},
    "raw_text": raw_text,
}

profile_id = save_profile(full_profile)
print(f"Saved as profile #{profile_id}")

Saved as profile #3


## Step 7 — Verify it saved and reads back correctly

In [15]:
saved = get_profile(profile_id)
print(json.dumps(saved, indent=2))

{
  "id": 3,
  "filename": "koti resumer.docx",
  "name": "M. Koteswara Rao\nPhone",
  "email": "mannemkoteswararao48@gmail.com",
  "phone": null,
  "location": null,
  "education": [],
  "skills": [
    "JavaScript",
    "Python",
    "SQL"
  ],
  "experience": [],
  "organizations": [
    "CSE",
    "AI & ML",
    "Object-Oriented Programming",
    "SQL",
    "Relational Database Management Systems",
    "JavaScript",
    "Student Database Management System",
    "SQL",
    "CRUD",
    "ORACLE\nPython",
    "Achievements & Activities\nActive",
    "Technical Clubs",
    "AI",
    "Strengths\nQuick"
  ],
  "certifications": [],
  "career_goal": {},
  "profile_completion": 37,
  "raw_text": "M. Koteswara Rao\nPhone: 70938 57339 | Email: mannemkoteswararao48@gmail.com\n\nCareer Objective\nEnthusiastic and detail-oriented B.Tech CSE (AI & ML) student with a strong foundation in programming, web development, and database management. Eager to contribute technical expertise, problem-solving